In [0]:
spark

# Bronze layer

In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
     # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df = spark.createDataFrame(data, columns)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("orders")
df.write.format("delta").mode("append").saveAsTable("orders")

# Silver layer

Data Cleaning

In [0]:
from pyspark.sql.functions import *
df=spark.read.table("orders")
df=df.filter(col("amount")>0)

In [0]:
df=df.fillna({
    "city":"Unknown",
    "amount":"0",
})

In [0]:
df=df.dropDuplicates()

In [0]:
df=df.withColumn("amount",col("amount").cast("int"))
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


latest data only

In [0]:
from pyspark.sql.window import Window
w=Window.partitionBy("order_id").orderBy(col("date").desc())
df=df.withColumn("row_number",row_number().over(w))
df=df.filter(col("row_number")==1).drop("row_number")
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


# Gold Layer


Requirement 1: Sales Analysis
1. Total sales by product
2. Total sales by category
3. Total sales by city

In [0]:
df.groupby("product").agg(sum("amount").alias("Total_sales")).show()

+----------+-----------+
|   product|Total_sales|
+----------+-----------+
|    Laptop|     105000|
|    Tablet|      22000|
|    Mobile|      33000|
|     Watch|       8000|
|Headphones|       3000|
+----------+-----------+



In [0]:
df.groupby("category").agg(sum("amount").alias("Total_sales")).show()

+-----------+-----------+
|   category|Total_sales|
+-----------+-----------+
|Electronics|     160000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
df.groupby("city").agg(sum("amount").alias("Total_sales")).show()

+---------+-----------+
|     city|Total_sales|
+---------+-----------+
|Hyderabad|      72000|
|    Delhi|      55000|
|  Chennai|      33000|
|   Mumbai|       8000|
|  Unknown|       3000|
+---------+-----------+



Requirement 2: Customer Insights
1. Number of orders per customer
2. Top customers based on spending

In [0]:
df.groupby("customer_id").agg(count("order_id").alias("Num_of_orders")).show()

+-----------+-------------+
|customer_id|Num_of_orders|
+-----------+-------------+
|       C001|            2|
|       C003|            1|
|       C002|            1|
|       C004|            1|
|       C005|            1|
|       C007|            1|
+-----------+-------------+



In [0]:
df.groupby("customer_id").agg(sum("amount").alias("Top_cust")).orderBy("Top_cust",ascending=False).show(1)

+-----------+--------+
|customer_id|Top_cust|
+-----------+--------+
|       C001|   72000|
+-----------+--------+
only showing top 1 row


# ANALYSIS

In [0]:
df.groupby("product").agg(count("product").alias("product-count")).orderBy("product-count",ascending=False).show()

+----------+-------------+
|   product|product-count|
+----------+-------------+
|    Laptop|            2|
|    Mobile|            2|
|    Tablet|            1|
|     Watch|            1|
|Headphones|            1|
+----------+-------------+



In [0]:
df.groupBy("customer_id",).agg(count(col("customer_id")).alias("top_customer")).orderBy("top_customer",ascending=False).show()

+-----------+------------+
|customer_id|top_customer|
+-----------+------------+
|       C001|           2|
|       C003|           1|
|       C002|           1|
|       C004|           1|
|       C005|           1|
|       C007|           1|
+-----------+------------+

